# <center> **RL Эксперименты с кастомной средой**

## <center> **1. Введение**

В рамках проекта рассматривается задача обучения агента с подкреплением в визуальной игровой среде **HeliRescue**, созданной на основе 2D‑игры на Pygame с управляемым вертолетом, падающим парашютистом и вражеским объектом alien. Агенту предоставляется доступ только к пиксельному представлению текущего кадра, а управление осуществляется через дискретный набор действий, имитирующих нажатия клавиш направления в исходной игре.

<img src="ScreenShots/screen1.png" alt="Game ScreenShot" style="width: 60%; height: 30%;">

**Основная цель проекта** — спроектировать и реализовать кастомную среду в формате Gymnasium, совместимую со Stable-Baselines3 и политиками типа CnnPolicy, провести серию экспериментов по обучению агента и проанализировать получающиеся стратегии поведения. Такой выбор постановки позволяет исследовать особенности визуального RL: влияние разрешения наблюдений, частоты действий (frame_skip) и формы функции награды на качество выученной политики в среде с непрерывным пространством состояний и дискретным управлением.

## <center> **2. Описание среды**

### **Пространство состояний**

Пространство состояний непрерывно и задается в виде $RGB$-изображения игрового кадра, сформированного с помощью pygame. На каждом шаге агент получает наблюдение $s_{t}$ в виде массива $uint8$ размера $(H, W, 3)$, где $H$ и $W$ - высота и ширина кадра после масштабирования масштабирования $84 X 84$ для использования в стандартной CnnPolicy в Stable-Baselines3. На изображении присутсивуют фон, вертолет, парашютист и пришелец, что делает задачу полностью наблюдаемой, но в визуальном, а не табличном виде.

Таким образом, состояние среды включает положения и скорости объектов, но агент не получает их в явном виде, а вынужден извлекать необходимые признаки (позиции, относительную геометрию, направление движения) напрямую из изображения с помощью сверточной нейронной сети.

### **Пространство действий**

Пространство действий дискретно и определяется как множество из пяти элементарных команд, соответствующих возможным перемещениям вертолета:

+ $0$ — отсутствие действия (NOOP, вертолет остается на месте по горизонтали и вертикали).
+ $1$ — движение влево при условии, что вертолет не выходит за левую границу игрового поля.
+ $2$ — движение вправо при условии, что вертолет не выходит за правую границу.
+ $3$ — движение вверх до верхней границы игрового окна.
+ $4$ — движение вниз до нижней границы игрового окна.

Наличие NOOP позволяет агенту при необходимости стабилизировать положение и не совершать лишних движений.

### **Функция награды**

Функция награды построена по мотивам игровой логики и поощряет стратегию действий, приводящую к спасению парашютиста и избеганию опасных ситуаций.

Столкновения:

+ Вертолет - парашютист: $+1$
+ Пришелец - парашютист: $-1$
+ Вертолет - пришелец: $-10$ и завершение эпизода.

Функция награды задается как скалярная величина:

$r_{t} = R(s_{t}, a_{t}, s_{t+1})$,

то есть как значение, зависящее от текущего состояния среды, действия агента и следующего состояния. В рассматриваемой задаче награда отражает основные действия игры.

$r_{t} = 1_{save}(t) - 1_{lose}(t) - 10 \cdot 1_{crash}(t) + 10 \cdot 1_{win}(t)$, где:

+ $1_{save}(t)$ - спасение парашютиста на шаге $t$
+ $1_{lose}(t)$ - перехват парашютиста пришельцем
+ $1_{crash}(t)$ - столкновение вертолета с пришельцем
+ $1_{win}(t)$ - достижение победы

### **Критерии успеха**

Критерием успеха в одном эпизоде является достижение целевого значения счета $+10$ до того, как счет опустится до $−10$ или будет превышен лимит по числу шагов эпизода. Эпизодическое завершение по достижению порога $+10$ интерпретируется как "победа" агента, так как накопленная серия успешных спасений перевешивает возможные ошибки и отражает устойчивую стратегию контроля вертолета.